In [1]:
import numpy as np
import bgshr
import pandas

In [2]:
ccds_lof_deciles = [f"ccds_lof_d{i}" for i in range(1, 12)]
phastcons_bins = [f"phastcons_{(5*i)}-{5*(i+1)}" for i in range(16)]
elem_types = ["ccds_merged", "promoters", "enhancers"] + ccds_lof_deciles + phastcons_bins
chroms = list(range(1, 23))
mut_models = ["carlson", "gnomad", "roulette"]

In [3]:
# Load U 
U_df = pandas.read_csv("../data/U_tbl.csv")

# Load DFE parameters
DFE_df = pandas.read_csv("../data/DFE_tbl.csv")
DFE_dict = {}
for elem_type in elem_types:
    dfe_type = next(iter(DFE_df[DFE_df["annotation"] == elem_type]["pdf"]))
    shape = next(iter(DFE_df[DFE_df["annotation"] == elem_type]["shape"]))
    scale = next(iter(DFE_df[DFE_df["annotation"] == elem_type]["scale_div2Ne"]))     
    if dfe_type == "gamma_neutral":
        p_neu = next(iter(DFE_df[DFE_df["annotation"] == elem_type]["p_neu"]))
        dfe = {"type": dfe_type, "shape": shape, "scale": scale, "p_neu": p_neu} 
    else:
        dfe = {"type": dfe_type, "shape": shape, "scale": scale} 
    DFE_dict[elem_type] = dfe

In [4]:
# Compute unlinked B effect produced by each chromosome/element type

cols = ["mut_model", "elem_type", "chrom", "U", "B"]
data = []

for chrom in chroms:
    for mut_model in mut_models:
        # Subset to chromosome/mutation model
        sub_df = U_df[(U_df["mut_model"] == mut_model) & (U_df["chrom"] == chrom)]
        for elem_type in elem_types:
            U = next(iter(sub_df[sub_df["elem_type"] == elem_type]["U"]))
            DFE = DFE_dict[elem_type]
            B = bgshr.ClassicBGS.unlinked_CBGS(U, DFE)
            row = [
                mut_model, 
                elem_type,
                chrom, 
                U, 
                B]
            data.append(row)

B_effect_df = pandas.DataFrame(data, columns=cols)

In [5]:
# Compute the B effect experienced by each chromosome

cons_models = {
    "merged_cds_regulatory": ["ccds_merged", "promoters", "enhancers"],
    "split_cds_regulatory": ccds_lof_deciles + ["promoters", "enhancers"],
    "merged_cds_phastcons": ["ccds_merged"] + phastcons_bins[:12],
    "split_cds_phastcons": ccds_lof_deciles + phastcons_bins[:12],
    # "merged_cds": ["ccds_merged"],
    # "split_cds": ccds_lof_deciles
}

cols = ["mut_model", "cons_model", "chrom", "B"]
data = []

for cons_model in cons_models:
    for mut_model in mut_models:
        for chrom in chroms:
            other_chroms = [i for i in chroms if i != chrom]
            elem_types = cons_models[cons_model]
            type_Bs = []
            for elem_type in elem_types:
                sub_df = B_effect_df[
                    (B_effect_df["mut_model"] == mut_model) 
                    & (B_effect_df["elem_type"] == elem_type)]
                df_Bs = list(sub_df["B"])
                df_chroms = list(sub_df["chrom"])
                type_B = np.prod([df_Bs[df_chroms.index(i)] for i in other_chroms])
                type_Bs.append(type_B)
            B = np.prod(type_Bs)
            row = [
                mut_model, 
                cons_model, 
                chrom, 
                B]
            data.append(row)
chrom_B_df = pandas.DataFrame(data, columns=cols)
chrom_B_df.to_csv("../models/B_unlinked_tbl.csv", index=False)

In [6]:
# Compute total unlinked BGS exerted by the whole autosome

cols = ["mut_model", "cons_model", "B"]
data = []

for cons_model in cons_models:
    for mut_model in mut_models:
        elem_types = cons_models[cons_model]
        # Get unlinked effect from each constraint class
        type_Bs = []
        for elem_type in elem_types:
            sub_df = B_effect_df[
                (B_effect_df["mut_model"] == mut_model) 
                & (B_effect_df["elem_type"] == elem_type)
            ]
            df_Bs = list(sub_df["B"])
            type_B = np.prod(df_Bs)
            type_Bs.append(type_B)
        model_B = np.prod(type_Bs)
        row = [
            mut_model, 
            cons_model, 
            model_B]
        data.append(row)
autosome_B_df = pandas.DataFrame(data, columns=cols)
autosome_B_df.to_csv("../models/autosome_wide_B_unlinked_tbl.csv", index=False)
print(autosome_B_df.to_string(index=False))

mut_model            cons_model        B
  carlson merged_cds_regulatory 0.943014
   gnomad merged_cds_regulatory 0.952204
 roulette merged_cds_regulatory 0.952288
  carlson  split_cds_regulatory 0.910649
   gnomad  split_cds_regulatory 0.920186
 roulette  split_cds_regulatory 0.922141
  carlson  merged_cds_phastcons 0.973599
   gnomad  merged_cds_phastcons 0.976537
 roulette  merged_cds_phastcons 0.977411
  carlson   split_cds_phastcons 0.940185
   gnomad   split_cds_phastcons 0.943702
 roulette   split_cds_phastcons 0.946469
